# Release16 Rulebase Reversal Checkpoint Readiness

Purpose: convert the `rulebase_reversal` target recipe into a BC/checkpoint-ready target contract and decide whether the next run is allowed to start a true checkpoint metric replay against the SOXLBacktest qlib buy/sell baseline.

Governance:
- Canonical baseline: retained SOXL harness model frontier.
- Direct comparison baseline: `release16_rulebase_reversal_target_recipe_v1`.
- Reference baseline: `soxlbacktest_qlib_walkforward_baseline_v1:wf_pass_ytd18_15_15_02`.
- Role: diagnose / handoff; no actor promotion or retention.
- Changed layer: `training_target_composition`, `model_selection`.
- Target windows: failed folds `wf_2020` and `wf_2023_2024`; `wf_2022` remains guard/no-overlay.
- Hypothesis: the rulebase reversal target rows can be merged into the source target contract so BC training actually consumes the intended higher participation target, without eval-window leakage.
- Acceptance: matched overlay rows, no eval overlap, no guard-fold writes, consumed target matches overlay target, enough fold coverage, and previous failed-fold context retained for the next checkpoint screen.


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_PATH = Path.cwd().resolve()
ROOT = NOTEBOOK_PATH
while ROOT != ROOT.parent and not (ROOT / 'src' / 'soxl_quant').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'soxl_quant').exists():
    raise RuntimeError(f'Could not locate repository root from {NOTEBOOK_PATH}')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.policy.policy_builder import state_objective_target_position_from_row

EXPERIMENT = 'release16_rulebase_reversal_checkpoint_readiness_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
SOURCE_TARGET = ROOT / 'runtime' / 'model_selection' / 'card' / 'release16_failedfold_participation_target_overlay_v1'
SOURCE_RECIPE = ROOT / 'runtime' / 'model_selection' / 'card' / 'release16_rulebase_reversal_target_recipe_v1'
SOURCE_CHECKPOINT = ROOT / 'runtime' / 'model_selection' / 'card' / 'release16_failedfold_overlay_checkpoint_screen_v1'

FAILED_FOLDS = ['wf_2020', 'wf_2023_2024']
GUARD_FOLDS = ['wf_2022']
MIN_ROWS_PER_FAILED_FOLD = 100
MIN_MEAN_TARGET_LIFT = 0.05
TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 20.0
CREATED_AT = datetime.now(timezone.utc).isoformat()
CARD.mkdir(parents=True, exist_ok=True)


def to_jsonable(value):
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_jsonable(v) for v in value]
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else float(value)
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return value


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(to_jsonable(payload), indent=2, sort_keys=True) + '\n', encoding='utf-8')


def fold_family_from_fold(value: object) -> str:
    text = str(value)
    if text.endswith('_train'):
        return text[:-6]
    return text


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## Load Source Contracts

The source target contract remains the complete train target surface. The rulebase reversal rows are an overlay on failed-fold train rows only.

In [2]:
source_targets = pd.read_csv(SOURCE_TARGET / 'combined_target_contract.csv')
recipe_contract = pd.read_csv(SOURCE_RECIPE / 'selected_rulebase_reversal_target_contract.csv')
recipe_summary = json.loads((SOURCE_RECIPE / 'summary.json').read_text(encoding='utf-8'))
checkpoint_summary = json.loads((SOURCE_CHECKPOINT / 'summary.json').read_text(encoding='utf-8'))
checkpoint_selection = pd.read_csv(SOURCE_CHECKPOINT / 'checkpoint_selection_by_fold.csv')

source_targets = source_targets.copy()
source_targets['fold_family'] = source_targets['fold'].map(fold_family_from_fold)
recipe_contract['fold_family'] = recipe_contract['fold'].map(fold_family_from_fold)

required_source_columns = {
    'datetime', 'fold', 'fold_family', 'window', 'teacher_position',
    'rebound_target_position', 'risk_target_position', 'objective_weight',
    'is_rebound_cluster', 'is_downtrend_state',
}
missing_source = sorted(required_source_columns - set(source_targets.columns))
if missing_source:
    raise ValueError(f'source target contract missing columns: {missing_source}')
required_recipe_columns = {'datetime', 'fold', 'fold_family', 'target_position', 'target_lift'}
missing_recipe = sorted(required_recipe_columns - set(recipe_contract.columns))
if missing_recipe:
    raise ValueError(f'rulebase recipe target contract missing columns: {missing_recipe}')

source_targets['_merge_key'] = source_targets['datetime'].astype(str) + '|' + source_targets['fold'].astype(str)
recipe_contract['_merge_key'] = recipe_contract['datetime'].astype(str) + '|' + recipe_contract['fold'].astype(str)
duplicate_recipe_keys = int(recipe_contract['_merge_key'].duplicated().sum())
matched_keys = set(source_targets['_merge_key']).intersection(set(recipe_contract['_merge_key']))
unmatched_recipe_rows = int(len(recipe_contract) - len(matched_keys))

source_targets.head(3)


,datetime,candidate,window,fold,target_scope,train_start,train_end,eval_start,eval_end,teacher_position,...,failedfold_participation_overlay_state,failedfold_participation_overlay_reason,failedfold_participation_overlay_floor,failedfold_participation_overlay_recipe_id,target_delta_vs_source,pre_overlay_objective_weight,failedfold_participation_target_position,participation_overlay_source_card,fold_family,_merge_key
0,2010-03-11,soxlbacktest_de_risk_teacher,wf_2020_train,wf_2020_train,fold_local,2010-03-11,2019-12-31,2020-01-01,2020-12-31,0.3,...,True,wf2020_release_rebound_floor,0.65,release16_failedfold_participation_floor_v1,0.25,6.0,0.65,rebound_state_factor_target_repair_screen_v1,wf_2020,2010-03-11|wf_2020_train
1,2010-03-12,soxlbacktest_de_risk_teacher,wf_2020_train,wf_2020_train,fold_local,2010-03-11,2019-12-31,2020-01-01,2020-12-31,0.3,...,False,source_contract,NaN,release16_failedfold_participation_floor_v1,0.00,1.0,0.30,rebound_state_factor_target_repair_screen_v1,wf_2020,2010-03-12|wf_2020_train
2,2010-03-15,soxlbacktest_de_risk_teacher,wf_2020_train,wf_2020_train,fold_local,2010-03-11,2019-12-31,2020-01-01,2020-12-31,0.3,...,False,source_contract,NaN,release16_failedfold_participation_floor_v1,0.00,1.0,0.30,rebound_state_factor_target_repair_screen_v1,wf_2020,2010-03-15|wf_2020_train


## Build BC-Consumed Overlay Contract

The prior source contract already stored the higher failed-fold participation target, but ordinary rows are consumed through `teacher_position`. This handoff explicitly sets the consumed target columns so `state_objective_target_position_from_row()` returns the rulebase reversal target on every overlay row.

In [3]:
overlay_by_key = recipe_contract.set_index('_merge_key').to_dict(orient='index')
combined = source_targets.copy()
combined['rulebase_reversal_overlay_state'] = combined['_merge_key'].isin(overlay_by_key)
combined['rulebase_reversal_overlay_source'] = ''
combined['rulebase_reversal_state_score'] = np.nan
combined['rulebase_reversal_target_lift'] = 0.0
combined['pre_rulebase_teacher_position'] = pd.to_numeric(combined['teacher_position'], errors='coerce')
combined['pre_rulebase_consumed_target_position'] = [
    state_objective_target_position_from_row(row) for row in combined.to_dict('records')
]

for idx, row in combined[combined['rulebase_reversal_overlay_state']].iterrows():
    overlay = overlay_by_key[str(row['_merge_key'])]
    target_position = float(overlay['target_position'])
    target_lift = float(overlay['target_lift'])
    combined.at[idx, 'teacher_position'] = target_position
    combined.at[idx, 'rebound_target_position'] = target_position
    combined.at[idx, 'risk_target_position'] = target_position
    if 'failedfold_participation_target_position' in combined.columns:
        combined.at[idx, 'failedfold_participation_target_position'] = target_position
    if 'current_target_position' in combined.columns:
        combined.at[idx, 'current_target_position'] = target_position
    combined.at[idx, 'rulebase_reversal_overlay_source'] = EXPERIMENT
    combined.at[idx, 'rulebase_reversal_state_score'] = float(overlay.get('state_score', np.nan))
    combined.at[idx, 'rulebase_reversal_target_lift'] = target_lift
    combined.at[idx, 'objective_weight'] = max(float(row.get('objective_weight', 1.0)), 6.0)

combined['post_rulebase_consumed_target_position'] = [
    state_objective_target_position_from_row(row) for row in combined.to_dict('records')
]
combined['rulebase_consumed_target_delta'] = (
    pd.to_numeric(combined['post_rulebase_consumed_target_position'], errors='coerce')
    - pd.to_numeric(combined['pre_rulebase_consumed_target_position'], errors='coerce')
)

overlay_rows = combined[combined['rulebase_reversal_overlay_state']].copy()
overlay_rows['expected_target_position'] = overlay_rows['_merge_key'].map(
    {key: float(value['target_position']) for key, value in overlay_by_key.items()}
)
overlay_rows['consumed_target_abs_error'] = (
    pd.to_numeric(overlay_rows['post_rulebase_consumed_target_position'], errors='coerce')
    - pd.to_numeric(overlay_rows['expected_target_position'], errors='coerce')
).abs()
consumed_target_mismatch_rows = int((overlay_rows['consumed_target_abs_error'] > 1e-9).sum())

combined_out = combined.drop(columns=['_merge_key'])
combined_out.to_csv(CARD / 'rulebase_reversal_bc_target_contract.csv', index=False)
overlay_rows.drop(columns=['_merge_key']).to_csv(CARD / 'rulebase_reversal_overlay_rows.csv', index=False)

overlay_rows[['datetime', 'fold', 'fold_family', 'pre_rulebase_consumed_target_position', 'post_rulebase_consumed_target_position', 'expected_target_position', 'rulebase_consumed_target_delta']].head()


,datetime,fold,fold_family,pre_rulebase_consumed_target_position,post_rulebase_consumed_target_position,expected_target_position,rulebase_consumed_target_delta
282,2011-04-21,wf_2020_train,wf_2020,0.4911,0.65,0.65,0.1589
283,2011-04-25,wf_2020_train,wf_2020,0.4911,0.65,0.65,0.1589
284,2011-04-26,wf_2020_train,wf_2020,0.4911,0.65,0.65,0.1589
285,2011-04-27,wf_2020_train,wf_2020,0.4911,0.65,0.65,0.1589
286,2011-04-28,wf_2020_train,wf_2020,0.4911,0.65,0.65,0.1589


## Readiness Checks

These checks are intentionally about training-contract readiness only. They do not claim the future actor will pass walk-forward; the next notebook must run that replay.

In [4]:
fold_summary = (
    overlay_rows.groupby('fold_family')
    .agg(
        overlay_rows=('datetime', 'count'),
        min_date=('datetime', 'min'),
        max_date=('datetime', 'max'),
        mean_target_lift=('rulebase_reversal_target_lift', 'mean'),
        median_target_lift=('rulebase_reversal_target_lift', 'median'),
        mean_consumed_delta=('rulebase_consumed_target_delta', 'mean'),
        max_consumed_error=('consumed_target_abs_error', 'max'),
    )
    .reset_index()
)
fold_summary.to_csv(CARD / 'target_position_delta_by_fold.csv', index=False)

bucket_source = source_targets.copy()
bucket_source['source_consumed_target_position'] = [
    state_objective_target_position_from_row(row) for row in bucket_source.to_dict('records')
]
bucket_source['target_bucket'] = pd.cut(
    pd.to_numeric(bucket_source['source_consumed_target_position'], errors='coerce'),
    bins=[-0.001, 0.25, 0.5, 0.65, 1.0],
    labels=['<=25%', '25-50%', '50-65%', '>65%'],
)
bucket_after = combined.copy()
bucket_after['target_bucket'] = pd.cut(
    pd.to_numeric(bucket_after['post_rulebase_consumed_target_position'], errors='coerce'),
    bins=[-0.001, 0.25, 0.5, 0.65, 1.0],
    labels=['<=25%', '25-50%', '50-65%', '>65%'],
)
bucket_delta = (
    pd.concat(
        [
            bucket_source.assign(version='source').groupby(['fold_family', 'target_bucket'], observed=False).size().rename('rows').reset_index(),
            bucket_after.assign(version='rulebase_reversal').groupby(['fold_family', 'target_bucket'], observed=False).size().rename('rows').reset_index(),
        ],
        ignore_index=True,
    )
)
bucket_delta.to_csv(CARD / 'target_position_bucket_delta.csv', index=False)

previous_context = checkpoint_selection.copy()
previous_context['failed_previous_candidate_gate'] = ~previous_context['selected_candidate_gate'].fillna(False).astype(bool)
previous_context.to_csv(CARD / 'previous_checkpoint_gap_context.csv', index=False)

overlay_folds = set(overlay_rows['fold_family'].astype(str))
failed_fold_coverage = {
    fold: int((overlay_rows['fold_family'].astype(str) == fold).sum())
    for fold in FAILED_FOLDS
}
guard_overlay_rows = int(overlay_rows['fold_family'].astype(str).isin(GUARD_FOLDS).sum())
eval_overlap_rows = int((~overlay_rows['window'].astype(str).str.endswith('_train')).sum())
mean_abs_lift = float(pd.to_numeric(overlay_rows['rulebase_reversal_target_lift'], errors='coerce').abs().mean()) if len(overlay_rows) else 0.0
previous_failed_folds = previous_context.loc[previous_context['failed_previous_candidate_gate'], 'fold'].astype(str).tolist()

checks = [
    {'check': 'recipe_rows_matched_source', 'pass': unmatched_recipe_rows == 0 and duplicate_recipe_keys == 0, 'value': len(overlay_rows), 'threshold': len(recipe_contract)},
    {'check': 'no_eval_overlap', 'pass': eval_overlap_rows == 0, 'value': eval_overlap_rows, 'threshold': 0},
    {'check': 'no_guard_fold_overlay', 'pass': guard_overlay_rows == 0, 'value': guard_overlay_rows, 'threshold': 0},
    {'check': 'consumed_target_matches_overlay', 'pass': consumed_target_mismatch_rows == 0, 'value': consumed_target_mismatch_rows, 'threshold': 0},
    {'check': 'failed_fold_wf_2020_coverage', 'pass': failed_fold_coverage.get('wf_2020', 0) >= MIN_ROWS_PER_FAILED_FOLD, 'value': failed_fold_coverage.get('wf_2020', 0), 'threshold': MIN_ROWS_PER_FAILED_FOLD},
    {'check': 'failed_fold_wf_2023_2024_coverage', 'pass': failed_fold_coverage.get('wf_2023_2024', 0) >= MIN_ROWS_PER_FAILED_FOLD, 'value': failed_fold_coverage.get('wf_2023_2024', 0), 'threshold': MIN_ROWS_PER_FAILED_FOLD},
    {'check': 'mean_abs_target_lift', 'pass': mean_abs_lift >= MIN_MEAN_TARGET_LIFT, 'value': mean_abs_lift, 'threshold': MIN_MEAN_TARGET_LIFT},
    {'check': 'previous_failed_fold_context_retained', 'pass': set(previous_failed_folds).issuperset(set(FAILED_FOLDS)), 'value': ','.join(previous_failed_folds), 'threshold': ','.join(FAILED_FOLDS)},
]
readiness_checks = pd.DataFrame(checks)
readiness_checks.to_csv(CARD / 'bc_readiness_checks.csv', index=False)

target_contract_gate = bool(readiness_checks.loc[readiness_checks['check'].isin(['recipe_rows_matched_source', 'no_eval_overlap', 'no_guard_fold_overlay', 'consumed_target_matches_overlay']), 'pass'].all())
bc_readiness_gate = bool(readiness_checks['pass'].all())
bc_screen_allowed_next = bool(target_contract_gate and bc_readiness_gate)
readiness_checks


,check,pass,value,threshold
0,recipe_rows_matched_source,True,569,569
1,no_eval_overlap,True,0,0
2,no_guard_fold_overlay,True,0,0
3,consumed_target_matches_overlay,True,0,0
4,failed_fold_wf_2020_coverage,True,424,100
5,failed_fold_wf_2023_2024_coverage,True,145,100
6,mean_abs_target_lift,True,0.187633,0.05
7,previous_failed_fold_context_retained,True,"wf_2020,wf_2023_2024","wf_2020,wf_2023_2024"


## Selection And Conclusion

This card should make the frontier point to an actual BC/checkpoint replay. Actor retention remains blocked until fresh fixed-split and walk-forward replay metrics prove the model clears the return and drawdown gates.

In [5]:
decision = (
    'rulebase_reversal_training_contract_ready_for_checkpoint_replay_no_promotion'
    if bc_screen_allowed_next
    else 'rulebase_reversal_training_contract_not_ready_no_promotion'
)
next_mechanism = (
    'run_rulebase_reversal_bc_checkpoint_metric_replay_with_soxlbacktest_baseline_and_walkforward_gates'
    if bc_screen_allowed_next
    else 'repair_rulebase_reversal_training_contract_before_checkpoint_replay'
)
primary_blocker = 'training_contract_ready' if bc_screen_allowed_next else 'training_contract_not_ready'

sidecars = {
    'rulebase_reversal_bc_target_contract': str((CARD / 'rulebase_reversal_bc_target_contract.csv').relative_to(ROOT)),
    'rulebase_reversal_overlay_rows': str((CARD / 'rulebase_reversal_overlay_rows.csv').relative_to(ROOT)),
    'target_position_delta_by_fold': str((CARD / 'target_position_delta_by_fold.csv').relative_to(ROOT)),
    'target_position_bucket_delta': str((CARD / 'target_position_bucket_delta.csv').relative_to(ROOT)),
    'previous_checkpoint_gap_context': str((CARD / 'previous_checkpoint_gap_context.csv').relative_to(ROOT)),
    'bc_readiness_checks': str((CARD / 'bc_readiness_checks.csv').relative_to(ROOT)),
}

selected_by_fold = checkpoint_summary.get('selected_by_fold', {})
metrics = checkpoint_summary.get('metrics', recipe_summary.get('metrics', []))
selection = {
    'experiment': EXPERIMENT,
    'created_at': CREATED_AT,
    'decision': decision,
    'canonical_baseline': 'retained_soxl_harness_model_frontier',
    'direct_comparison_baseline': 'release16_rulebase_reversal_target_recipe_v1',
    'reference_baseline': recipe_summary.get('reference_baseline', 'soxlbacktest_qlib_walkforward_baseline_v1:wf_pass_ytd18_15_15_02'),
    'source_target_experiment': 'release16_failedfold_participation_target_overlay_v1',
    'source_recipe_experiment': 'release16_rulebase_reversal_target_recipe_v1',
    'source_checkpoint_context': 'release16_failedfold_overlay_checkpoint_screen_v1',
    'role': 'diagnose',
    'changed_layer': ['training_target_composition', 'model_selection'],
    'failed_folds': FAILED_FOLDS,
    'guard_folds': GUARD_FOLDS,
    'target_contract_rows': int(len(combined_out)),
    'overlay_rows': int(len(overlay_rows)),
    'failed_fold_coverage': failed_fold_coverage,
    'mean_abs_target_lift': mean_abs_lift,
    'consumed_target_mismatch_rows': consumed_target_mismatch_rows,
    'target_contract_gate': target_contract_gate,
    'bc_readiness_gate': bc_readiness_gate,
    'bc_screen_allowed_next': bc_screen_allowed_next,
    'next_training_allowed_after_target_revision': bc_screen_allowed_next,
    'target_revision_ready': bc_screen_allowed_next,
    'training_contract_ready': bc_screen_allowed_next,
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'primary_blocker': primary_blocker,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN_PP,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'metrics': metrics,
    'selected_by_fold': selected_by_fold,
    'next_mechanism': next_mechanism,
    'sidecars': sidecars,
    'summary_points': [
        f"Merged {len(overlay_rows)} rulebase reversal target rows into the source BC target contract.",
        f"Failed-fold coverage: wf_2020={failed_fold_coverage.get('wf_2020', 0)}, wf_2023_2024={failed_fold_coverage.get('wf_2023_2024', 0)}.",
        f"Mean absolute consumed target lift is {mean_abs_lift:.3f}; consumed target mismatches: {consumed_target_mismatch_rows}.",
        f"Training contract ready: {bc_screen_allowed_next}; no actor promotion or retention from this card.",
    ],
}
summary = {
    **selection,
    'readiness_checks': readiness_checks.to_dict(orient='records'),
    'fold_summary': fold_summary.to_dict(orient='records'),
    'previous_failed_folds': previous_failed_folds,
    'conclusion': 'Training contract is ready for a true BC/checkpoint metric replay only if bc_screen_allowed_next is true.',
}

write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)

conclusion_lines = [
    f'# {EXPERIMENT}',
    '',
    f'- Decision: `{decision}`.',
    f'- Overlay rows: `{len(overlay_rows)}`; target contract rows: `{len(combined_out)}`.',
    f'- Failed-fold coverage: `wf_2020={failed_fold_coverage.get("wf_2020", 0)}`, `wf_2023_2024={failed_fold_coverage.get("wf_2023_2024", 0)}`.',
    f'- Mean absolute consumed target lift: `{mean_abs_lift:.3f}`; consumed target mismatches: `{consumed_target_mismatch_rows}`.',
    f'- Gates: target contract `{target_contract_gate}`, BC readiness `{bc_readiness_gate}`, BC/checkpoint screen allowed next `{bc_screen_allowed_next}`.',
    f'- Next action: `{next_mechanism}`.',
    '',
    'No actor is promoted or retained. The next card must run fresh BC/checkpoint replay metrics against the SOXLBacktest qlib buy/sell baseline and the walk-forward retention gates.',
]
conclusion_text = '\n'.join(conclusion_lines) + '\n'
(CARD / 'conclusion.md').write_text(conclusion_text, encoding='utf-8')
display(Markdown(conclusion_text))


# release16_rulebase_reversal_checkpoint_readiness_v1

- Decision: `rulebase_reversal_training_contract_ready_for_checkpoint_replay_no_promotion`.
- Overlay rows: `569`; target contract rows: `3731`.
- Failed-fold coverage: `wf_2020=424`, `wf_2023_2024=145`.
- Mean absolute consumed target lift: `0.188`; consumed target mismatches: `0`.
- Gates: target contract `True`, BC readiness `True`, BC/checkpoint screen allowed next `True`.
- Next action: `run_rulebase_reversal_bc_checkpoint_metric_replay_with_soxlbacktest_baseline_and_walkforward_gates`.

No actor is promoted or retained. The next card must run fresh BC/checkpoint replay metrics against the SOXLBacktest qlib buy/sell baseline and the walk-forward retention gates.
